# Chapter 7 — Not All Tokens Are Equal

## Question

**Under context pressure, should every item be treated the same way?**

Falsifiable version: if items carry different retention contracts, does at least one uniform policy violate at least one contract? If yes, pressure cannot be handled by treating tokens as interchangeable.

This notebook classifies only. It performs no pruning, no compaction, no externalisation. Later notebooks act; this one decides what would be legal.

## Setup — a pressured bundle of nine items

Each item carries the chapter's six properties: exactness, authority, recoverability, relevance, volatility, token cost. Token counts are fixtures. No single `importance_score` exists anywhere below, by design.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class RetentionClass(Enum):
    PIN = 'PIN'
    COMPRESSIBLE = 'COMPRESSIBLE'
    EXTERNALIZABLE = 'EXTERNALIZABLE'
    REFETCHABLE = 'REFETCHABLE'
    DISCARDABLE = 'DISCARDABLE'

@dataclass(frozen=True)
class ContextItem:
    id: str
    label: str
    exactness: str       # 'REQUIRED' | 'PARTIAL' | 'LOW'
    authority: str       # 'governs' | 'constrains' | 'none'
    recoverable: bool
    recovery: str        # path, cost, or 'none'
    relevance: str       # 'active' | 'conditional' | 'spent'
    volatility: str      # 'stable' | 'decays' | 'high'
    tokens: int
    age_turns: int
    retention_class: RetentionClass

ITEMS = [
    ContextItem('rule', 'Project safety rule', 'REQUIRED', 'governs', False, 'none: only copy', 'active', 'stable', 60, 40, RetentionClass.PIN),
    ContextItem('port', 'Deploy port value', 'REQUIRED', 'constrains', False, 'none: only copy', 'active', 'stable', 20, 38, RetentionClass.PIN),
    ContextItem('tests', 'Test-result summary', 'PARTIAL', 'none', True, 'costly: re-run suite', 'active', 'high', 400, 2, RetentionClass.COMPRESSIBLE),
    ContextItem('log', 'Compiler log', 'LOW', 'none', True, 'costly: rebuild', 'conditional', 'high', 18000, 1, RetentionClass.COMPRESSIBLE),
    ContextItem('adr', 'Architecture decision', 'PARTIAL', 'constrains', True, 'stored reference ADR-007', 'active', 'stable', 500, 35, RetentionClass.EXTERNALIZABLE),
    ContextItem('src', 'Source file contents', 'LOW', 'none', True, 'cheap: re-read src/checkout.py', 'active', 'decays', 9000, 5, RetentionClass.REFETCHABLE),
    ContextItem('verdict', 'Unique migration verdict', 'PARTIAL', 'constrains', False, 'none: no other record', 'active', 'stable', 1200, 3, RetentionClass.COMPRESSIBLE),
    ContextItem('hypo', 'Rejected hypothesis', 'LOW', 'none', True, 'moot: resolved', 'spent', 'stable', 300, 30, RetentionClass.DISCARDABLE),
    ContextItem('dup', 'Duplicate observation', 'LOW', 'none', True, 'via original span', 'spent', 'stable', 1200, 0, RetentionClass.DISCARDABLE),
]
by_id = {x.id: x for x in ITEMS}
TOTAL = sum(x.tokens for x in ITEMS)
print(f'{len(ITEMS)} items, {TOTAL} fixture tokens under pressure.')

## Baseline — Policy A keeps everything

The usable budget is 20,000 tokens. Keeping everything is always contract-legal and sometimes physically impossible.

In [ ]:
USABLE = 20000
print(f'bundle: {TOTAL} tokens; usable budget: {USABLE}')
print(f'Policy A (keep everything): legal=yes, feasible={TOTAL <= USABLE}')
assert TOTAL == 30680
assert TOTAL > USABLE, 'pressure must be real or the policies decide nothing'

## Intervention — four naive policies judged against the contracts

Legality rules used below (policy-validity demonstration, not behaviour): summarising an item whose exactness is REQUIRED is ILLEGAL; removing an item is legal only for REFETCHABLE or DISCARDABLE classes; age-ordered removal is judged by what it actually drops.

In [ ]:
def legality(item, transform):
    if transform == 'summarise':
        return 'LEGAL' if item.exactness != 'REQUIRED' else 'ILLEGAL'
    if transform == 'remove':
        return ('LEGAL' if item.retention_class in (RetentionClass.REFETCHABLE, RetentionClass.DISCARDABLE)
                else 'ILLEGAL')
    return 'LEGAL'  # keep

# B removes the two oldest; D keeps the two newest (removes the rest); C summarises all.
oldest = sorted(ITEMS, key=lambda x: -x.age_turns)[:2]
newest = sorted(ITEMS, key=lambda x: x.age_turns)[:2]
POLICIES = {
    'B remove-oldest-first': {x.id: ('remove' if x in oldest else 'keep') for x in ITEMS},
    'C summarise-everything': {x.id: 'summarise' for x in ITEMS},
    'D keep-newest-first': {x.id: ('keep' if x in newest else 'remove') for x in ITEMS},
    'E typed-retention': {'rule': 'keep', 'port': 'keep', 'tests': 'summarise', 'log': 'summarise',
                          'adr': 'keep', 'src': 'remove', 'verdict': 'summarise',
                          'hypo': 'remove', 'dup': 'remove'},
}
all_valid = {}
for pname, transforms in POLICIES.items():
    bad = [(i, transforms[i]) for i in transforms
           if legality(by_id[i], transforms[i]) == 'ILLEGAL']
    all_valid[pname] = not bad
    print(f"{pname}: {'VALID for all items' if not bad else 'VIOLATES ' + ', '.join(f'{i} ({t})' for i, t in bad)}")
assert not all_valid['B remove-oldest-first']
assert not all_valid['C summarise-everything']
assert not all_valid['D keep-newest-first']
assert all_valid['E typed-retention']
assert not any(p.startswith(('B', 'C', 'D')) and v for p, v in all_valid.items()), \
    'no uniform policy is valid for every item'

## The importance-score trap — one dimension cannot carry two decisions

Hypothetical scores, invented for this cell only: the source file scores 95 (vital, refetchable), the port value scores 5 (trivial, exact). A keep-top-by-score policy with summarise-the-rest destroys the port while insuring the disk.

In [ ]:
hypothetical_importance = {'src': 95, 'port': 5}
print('score order: src (95) first, port (5) last')
print(f"score policy summarises port: {legality(by_id['port'], 'summarise')} (exactness REQUIRED)")
print(f"score policy pins src: wasteful but {legality(by_id['src'], 'keep')} (disk already holds it)")
assert legality(by_id['port'], 'summarise') == 'ILLEGAL'
print('Important does not mean keep; trivial does not mean summarise. "Important" never states the legal transform.')

## Recoverability — same cost, different guarantees

The duplicate observation and the unique verdict both cost 1,200 tokens. One has a recovery path; the other is the only copy.

In [ ]:
a, b = by_id['dup'], by_id['verdict']
print(f"{a.label}: {a.tokens} tokens, recoverable={a.recoverable} -> removal {legality(a, 'remove')}")
print(f"{b.label}: {b.tokens} tokens, recoverable={b.recoverable} -> removal {legality(b, 'remove')}")
assert a.tokens == b.tokens
assert a.recoverable is True and b.recoverable is False
assert legality(a, 'remove') == 'LEGAL' and legality(b, 'remove') == 'ILLEGAL'

## Observation — the compiler-view record

No weights, no score. Merely the information later mechanisms may consult, and must state they consulted.

In [ ]:
for x in ITEMS:
    print(f"{x.id:8s} cost={x.tokens:6d} authority={x.authority:10s} scope=task freshness={x.volatility:6s} "
          f"recoverability={x.recovery:32s} exactness={x.exactness:8s} class={x.retention_class.value}")
assert by_id['rule'].retention_class is RetentionClass.PIN
assert by_id['src'].recoverable is True

## Try it

1. Reclassify `adr` as PIN and re-run the policy matrix — Policy E becomes invalid until its transform for `adr` changes to keep.
2. Set `log` exactness to REQUIRED and watch Policy C gain a third violation.
3. Lower USABLE to 10,000: Policy A stays legal and infeasible, which is why legality and feasibility are separate verdicts.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(legality(by_id['adr'], 'remove'))

## What this demonstrates

- Context pressure cannot be handled correctly while treating every token as interchangeable: every uniform policy above violates at least one retention contract.
- Classification determines which transformations are legal; it performs none of them.
- Recoverability splits items more decisively than cost: equal tokens, opposite legal choices.

## What this does not demonstrate

- That these five classes are optimal.
- That typed retention improves model behaviour: no model ran.
- That any item should yet be deleted or summarised.
- That recoverability equals relevance, or that age implies discardability.
- That importance is useless in every possible system: it is refused here as the primitive, not as an input.

## Connection to the chapter

Retention is the policy input; transformation is someone else's machinery. With classification established, the pressure-relief question returns transformed — and one deferred alternative deserves its hearing first:

> If capacity pressure causes these difficult choices, perhaps the simplest answer is to make capacity much larger.

That is Chapter 8.